In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from qhdopt import QHD

In [3]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到

# 2bus model
Sbase_2 = 10.0
buses_2 = {
    1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
    2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
}
lines_2 = {
    1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase_2],
}
gens_2 = {
    1: [1, 0.0 / Sbase_2, 20.0 / Sbase_2, -20.0 / Sbase_2, 100.0 / Sbase_2, 0.00375, 2.0, 0.0],
}

model = SympyACOPFModel(Sbase = Sbase_2, buses=buses_2, lines=lines_2, gens=gens_2)

In [4]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()

rho = 10.0
alpha = 5e-1   # 对偶步长尽量小一点
max_outer = 50
tol = 1e-4

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

    option = 1
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)

        qhd_model.simbi_setup(
            resolution=6,
            agents=1024,
            max_steps=1000,
            embedding_scheme="unary",
            post_processing_method='TNC',
            verbose=True
        )

        response = qhd_model.optimize(verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1e4
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (7) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (6) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")


===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.6848056316375732
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0082	-0.0014	0.2934	0.0977	1.0000	0.0000	0.0000	0.0000
2	1.0064	-0.0100	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.2934	0.0977		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0054	-0.0462	0.0203	0.0017


Total Ploss: -0.0408
Total Qloss: 0.0220
Total Load Supplied: 111.4164%
||h(x)|| = 0.38535346823990796
[rho-check] ||h_old||=4.939e-01, ||h_new||=3.854e-01, rho=10
Increasing rho to 20.0
Constraint check: False

--- Outer Iteration 1 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4704256057739258
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0160	0.0057	0.2902	0.0943	1.0000	0.0000	0.0000	0.0000
2	0.9978	-0.0251	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.2902	0.0943		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0086	-0.1802	0.0235	-0.0495


Total Ploss: -0.1716
Total Qloss: -0.0260
Total Load Supplied: 153.9349%
||h(x)|| = 0.25218408491590544
[rho-check] ||h_old||=3.854e-01, ||h_new||=2.522e-01, rho=20
Increasing rho to 40.0
Constraint check: False

--- Outer Iteration 2 ---


...

backend time consumption: 0.45006537437438965
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.8495	0.2349	0.5472	0.2384	1.0000	0.0000	0.0000	0.0000
2	0.8664	0.1095	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.5472	0.2384		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.5733	-0.5617	0.3064	0.1418


Total Ploss: 0.0116
Total Qloss: 0.4482
Total Load Supplied: 178.5337%
||h(x)|| = 0.858321742988933
[rho-check] ||h_old||=1.099e+00, ||h_new||=8.583e-01, rho=1.02e+04
Constraint check: False

--- Outer Iteration 48 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4331986904144287
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.8305	0.4362	1.0156	0.2287	1.0000	0.0000	0.0000	0.0000
2	0.8627	0.3006	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			1.0156	0.2287		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.5476	-0.6618	0.2521	0.0914


Total Ploss: -0.1142
Total Qloss: 0.3435
Total Load Supplied: 376.5772%
||h(x)|| = 0.8437117984234994
[rho-check] ||h_old||=8.583e-01, ||h_new||=8.437e-01, rho=1.02e+04
Constraint check: False

--- Outer Iteration 49 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4336080551147461
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.1000	-0.4486	0.0186	-0.2369	1.0000	0.0000	0.0000	0.0000
2	0.9912	-0.5841	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.0186	-0.2369		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.5581	-1.0440	-0.2457	0.0999


Total Ploss: -0.4859
Total Qloss: -0.1457
Total Load Supplied: 168.1523%
||h(x)|| = 1.7064370009024696
[rho-check] ||h_old||=8.437e-01, ||h_new||=1.706e+00, rho=1.02e+04
Constraint check: False

===== End Loop =====



In [5]:
response.refined_minimizer


array([ 0.01855442, -0.23693823,  1.1       ,  0.99121085, -0.44858127,
       -0.58409621,  0.92221018,  0.87582285,  0.55810253, -0.24569293,
        0.03649866])

In [6]:
qhd_model.func

0.001875*P_G0**2 + 4347.81907162983*P_G0 - 1455.82494810148*P_ij0 + 1186.0608267216*Q_G0 + 1094.48469967555*Q_ij0 + 458.872137486643*S_tot_sq0 + 5120.0*V_I0**2 + 5633.65692927645*V_I0 + 7792.4519145672*V_I1 - 26014.5795186493*V_R0 + 17487.0031224847*V_R1 + 2244.05885318018*V_sq0 + 3221.15129908608*V_sq1 + 750.0*(P_G0 - 1.01555743695228)**2 + 750.0*(P_ij0 - 0.547644651861656)**2 + 750.0*(Q_G0 - 0.228710075773273)**2 + 750.0*(Q_ij0 - 0.252128065946528)**2 + 750.0*(S_tot_sq0 - 0.452865147099852)**2 + 750.0*(V_I0 - 0.436220569923186)**2 + 750.0*(V_I1 - 0.300576431404918)**2 + 750.0*(V_R0 - 0.830505034891655)**2 + 750.0*(V_R1 - 0.862702011170644)**2 + 750.0*(V_sq0 - 1.11606594602763)**2 + 750.0*(V_sq1 - 1.06506034305924)**2 + 5120.0*(-1.09528930372331*P_ij0 - 0.504256131893057*Q_ij0 + 1.0*S_tot_sq0 + 0.363483226350611)**2 + 5120.0*(-0.872441139846373*V_I0 - 1.66101006978331*V_R0 + 1.0*V_sq0 + 0.880026998604498)**2 + 5120.0*(-0.601152862809836*V_I1 - 1.72540402234129*V_R1 + 1.0*V_sq1 + 0.834

In [7]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.3] [0.  0.1] (P_G0,) (Q_G0,)
[P_G0, Q_G0, V_R0, V_R1, V_I0, V_I1, V_sq0, V_sq1, P_ij0, Q_ij0, S_tot_sq0]
[[0.0, 2.0], [-2.0, 10.0], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [-3.0, 3.0], [-3.0, 3.0], [0.0, 9.0]]
